In [4]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env file in this folder

# Or set it directly here for quick demos:
# os.environ["GROQ_API_KEY"] = "your-key-here"

assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in a .env file or above"
print("API key loaded.")

API key loaded.


In [5]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"  # adjust if needed

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

C:\Windows\Temp\ipykernel_5040\1501032524.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\AI projects\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # ~150 words per chunk
    chunk_overlap=100,    # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}  (from {len(pages)} pages)")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print("\n--- Example chunk ---")
print(chunks[5].page_content)

Total chunks: 37  (from 9 pages)
Avg chunk length: 504 chars

--- Example chunk ---
Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 dBm is poor. If signal is weak, moving closer to a window or to a higher floor often helps.


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model (downloads ~90 MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in Chroma (in memory)...")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading embedding model (downloads ~90 MB on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5006.35it/s]


Embedding all chunks and storing in Chroma (in memory)...
Vector store ready. 37 vectors stored.


In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call c

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    reasoning_format="parsed",
)

# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [13]:
import requests

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}"}
)
models = [m["id"] for m in resp.json()["data"]]
print("\n".join(sorted(models)))

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


In [10]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: Based on the provided context, here is how international roaming works and the expected charges:

**How It Works**
*   **Connection:** When you travel outside your home network's coverage area, your device connects to a partner network in the visited country.
*   **Authentication & Billing:** The visited network authenticates you via inter-operator signalling protocols (SS7 or Diameter). Your home network validates your subscription and authorizes service. All voice, data, and SMS traffic is tunnelled back to the home network for billing.
*   **Latency:** This tunnelling process adds latency compared to using a local network.

**Expected Charges (Roaming Zones)**
Charges depend on which "roaming zone" the country you are visiting falls into, based on inter-operator agreements and cost structures:
*   **Zone A (EU, UK, Australia, New Zealand):** Lowest roaming rates.
*   **Zone B (USA, Canada, Japan, Singapore)

In [ ]:
print("Telecom RAG Assistant — type 'quit' to exit\n")

while True:
    question = input("Your question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    print("\nAnswer:")
    for chunk in chain.stream(question):
        print(chunk, end="", flush=True)
    print("\n")

Telecom RAG Assistant — type 'quit' to exit


Answer:
Based on the provided context, **Zone A** has the cheapest roaming plans.

The text states:
*   **Zone A** (EU, UK, Australia, New Zealand) has the **lowest** roaming rates.
*   **Zone B** (USA, Canada, Japan, Singapore) has **moderate** rates.
*   **Zone C** (Rest of World) has the **highest** per-MB and per-minute charges.


Answer:
It seems like your message might have been cut off or sent accidentally. How can I help you today?



In [1]:
debug_question = "What security measures protect against SIM swap fraud?"

docs = retriever.invoke(debug_question)
print(f"Question: {debug_question}")
print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs, 1):
    print(f"{'='*60}")
    print(f"Chunk {i} (page {doc.metadata.get('page', '?')})")
    print(f"{'='*60}")
    print(doc.page_content)
    print()

print("\nFinal Answer:")
print(chain.invoke(debug_question))

NameError: name 'retriever' is not defined